# Brain Fresh — export TRIBE analysis on Colab (GPU)

Use this when local (Mac) inference is too slow. You get a JSON file that matches the Brain Fresh API.

**Workflow**
1. Runtime → Change runtime type → **GPU** (T4/L4/A100).
2. Run setup cells below (installs the backend + TRIBE; first run takes several minutes).
3. Upload your video (same file you will use in the app). **Tip:** macOS screen recordings often have spaces and a special Unicode space before `AM/PM` in the filename; the export cell copies your file to `/content/brain_input.mp4` to avoid path issues.
4. Set `BRAIN_FRESH_VIDEO_ID` to the **8-character id** shown in the web app right after upload (open DevTools → Network → `upload` response, or use the import flow below).
5. Run the **export** cell (loads the script in-process — **not** `subprocess` — so errors print as a normal traceback). tqdm shows progress; then download `analysis_export.json`.
6. Locally: add `ALLOW_ANALYSIS_IMPORT=1` to `backend/.env`, restart the API, then:
   `curl -X POST "http://localhost:8000/api/video/import-result/YOUR_ID" -H "Content-Type: application/json" -d @analysis_export.json`
7. In the app, if processing failed, use **Load analysis JSON** and pick the file — or refresh and open results if you used `curl`.

Optional: `huggingface-cli login` or set `HF_TOKEN` for higher Hub rate limits.

**If you see `ModuleNotFoundError: tribev2`:** use **`%pip`** (not `!pip`) so installs hit the notebook kernel.

**If you see `No matching distribution found for tribev2`:** TRIBE is **not on PyPI** — it only exists as a **GitHub** package. The install cell installs that repo **first**, then `brain-fresh-backend` in editable mode.

In [ ]:
!nvidia-smi

In [ ]:
# Clone your fork (replace URL) or upload a zip of brain_fresh and %cd into it
# !git clone https://github.com/YOUR/brain_fresh.git
%cd /content/brain_fresh/backend

# tribev2 is NOT on PyPI — Colab pip may look there first when resolving -e . alone
%pip install -U pip setuptools wheel
%pip install "git+https://github.com/facebookresearch/tribev2.git"

# %pip = this notebook's Python (not !pip)
%pip install -e .

import tribev2

print("tribev2 OK →", tribev2.__file__)

In [ ]:
from google.colab import files
import pathlib

up = files.upload()
video_path = pathlib.Path(list(up.keys())[0]).resolve()
print("Video:", video_path)

In [ ]:
# Export runs IN THIS NOTEBOOK (no subprocess) so you always see the real Python traceback.
# Replace xxxxxxxx with your 8-char video id from the app.
BRAIN_FRESH_VIDEO_ID = "xxxxxxxx"

import importlib.util
import pathlib
import shutil
from google.colab import files

BACKEND = pathlib.Path("/content/brain_fresh/backend")
SCRIPT = BACKEND / "scripts" / "export_tribe_analysis.py"
if not SCRIPT.is_file():
    raise FileNotFoundError(
        f"Missing {SCRIPT}. Clone the repo to /content/brain_fresh and run the install cell."
    )

safe_video = pathlib.Path("/content/brain_input.mp4")
shutil.copy2(video_path, safe_video)
print("Using:", safe_video)

spec = importlib.util.spec_from_file_location("brain_export", SCRIPT)
mod = importlib.util.module_from_spec(spec)
assert spec.loader is not None
spec.loader.exec_module(mod)

out = pathlib.Path("/content/analysis_export.json")
mod.run_export_for_notebook(
    safe_video,
    out,
    BRAIN_FRESH_VIDEO_ID,
    show_progress=True,
)
files.download(str(out))